# Day 18 — Merge, Join & Concat
### Python for Data Science · Module 1 · Topic 1.17

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | The idea of a join | 15 min |
| 2 | **The four joins** | 30 min |
| 3 | Merge in practice | 20 min |
| 4 | **Duplicate keys & concat** | 20 min |
| 5 | Mini build: three tables into one report | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **The question that matters in every merge.** Not *"which rows match?"* — pandas handles
> that. The question is what should happen to the rows that **don't**. Dropping a customer
> with no orders, or keeping them with a blank total, are different business answers to the
> same data. **The join type is that decision, written in code.**

In [1]:
import pandas as pd
import numpy as np

students = pd.DataFrame({
    "id":   [1, 2, 3, 4],
    "name": ["Ravi", "Sara", "Amit", "Neha"],
    "city": ["Pune", "Mumbai", "Delhi", "Pune"],
})
marks = pd.DataFrame({
    "id":     [1, 2, 3, 5],
    "python": [88, 91, 45, 70],
})

print(students)
print()
print(marks)
print()
print("id 4 has no marks.  id 5 has no student.")

   id  name    city
0   1  Ravi    Pune
1   2  Sara  Mumbai
2   3  Amit   Delhi
3   4  Neha    Pune

   id  python
0   1      88
1   2      91
2   3      45
3   5      70

id 4 has no marks.  id 5 has no student.


---
# 1. The idea of a join

Storing names once in a student table, rather than repeating them beside every mark, is how
databases avoid duplication and contradictions. The price is that answering a real question
needs two tables put back together.

You met this on **Day 9**, when a CSV of cities sat beside a CSV of marks and you built a
lookup dictionary by hand. Today pandas does it properly.

In [ ]:
students.merge(marks, on="id")

# Three rows. ids 4 and 5 both vanished - because the DEFAULT is an inner join.

---
# 2. The four joins

| `how` | Keeps | Unmatched become | Use it when |
|---|---|---|---|
| `"inner"` | rows in **both** | dropped entirely | you need complete records only |
| `"left"` | all left rows | `NaN` in the right columns | the left table is your population |
| `"right"` | all right rows | `NaN` in the left columns | rare — swap the tables instead |
| `"outer"` | everything | `NaN` on whichever side is missing | auditing what is in one and not the other |

In [15]:
for how in ["inner", "left", "right", "outer"]:
    m = students.merge(marks, on="id", how=how)
    print(f"{how:6}  rows={len(m)}  ids={sorted(m['id'].tolist())}")

inner   rows=3  ids=[1, 2, 3]
left    rows=4  ids=[1, 2, 3, 4]
right   rows=4  ids=[1, 2, 3, 5]
outer   rows=5  ids=[1, 2, 3, 4, 5]


In [20]:
# left - every student survives, Neha with no mark
students.merge(marks, on="id", how="outer")

,id,name,city,python
0,1,Ravi,Pune,88.0
1,2,Sara,Mumbai,91.0
2,3,Amit,Delhi,45.0
3,4,Neha,Pune,NaN
4,5,NaN,NaN,70.0


In [ ]:
# outer - everything from both sides
students.merge(marks, on="id", how="outer")

## 2.1 ⚠️ An unmatched row changes the dtype

In [ ]:
before = marks["python"].dtype
after  = students.merge(marks, on="id", how="left")["python"].dtype

print("before the merge:", before)
print("after  the merge:", after)

# Day 14 again: an integer column cannot hold NaN, so the moment one
# unmatched row appears, the whole column becomes float.
#
# If your IDs or counts suddenly print as 88.0, an unmatched merge is a
# likely cause.

## 2.2 `indicator=True` tells you exactly what happened

In [ ]:
check = students.merge(marks, on="id", how="outer", indicator=True)
print(check[["id", "name", "python", "_merge"]])
print()
print(check["_merge"].value_counts())

> **A habit worth adopting:** run an outer join with `indicator=True` **before** choosing
> your real join. It answers *"how many rows are in one table and not the other?"* in one
> step — and the answer often changes which join you actually want.

---
# 3. Merge in practice

## 3.1 Keys with different names

In [4]:
s2 = students.rename(columns={"id": "student_id"})

joined = s2.merge(marks, left_on="student_id", right_on="id",how = "outer")
print(joined.columns.tolist())

# You get BOTH key columns. Drop one afterwards:
print(joined.drop(columns="id").columns.tolist())
print(joined)

['student_id', 'name', 'city', 'id', 'python']
['student_id', 'name', 'city', 'python']
   student_id  name    city   id  python
0         1.0  Ravi    Pune  1.0    88.0
1         2.0  Sara  Mumbai  2.0    91.0
2         3.0  Amit   Delhi  3.0    45.0
3         4.0  Neha    Pune  NaN     NaN
4         NaN   NaN     NaN  5.0    70.0


## 3.2 ⚠️ Overlapping column names

In [22]:
a = pd.DataFrame({"id": [1, 2], "score": [10, 20]})
b = pd.DataFrame({"id": [1, 2], "score": [99, 98]})

print(a.merge(b, on="id"))          # score_x and score_y - unhelpful
print()
print(a.merge(b, on="id", suffixes=("_table_a", "_table_b")))

   id  score_x  score_y
0   1       10       99
1   2       20       98

   id  score_table_a  score_table_b
0   1             10             99
1   2             20             98


## 3.3 merge, join, and the family of names

| | Works on | Default `how` |
|---|---|---|
| `a.merge(b, on="id")` | a **column** | `inner` |
| `left.join(right)` | the **index** | `left` |
| `pd.merge(a, b, on="id")` | same as `a.merge(...)` | `inner` |

**Learn `merge` properly and use it everywhere.** You can always `set_index` first if you
want index behaviour. Having three near-synonyms half-known is worse than one habit.

In [ ]:
left  = students.set_index("id")
right = marks.set_index("id")

print(left.join(right))        # index-based, keeps all left rows

---
# 4. Duplicate keys & concat

## 4.1 ⚠️ The bug that multiplies your data

In [6]:
orders    = pd.DataFrame({"cust": [1, 1, 2], "amt":  [10, 20, 30]})
customers = pd.DataFrame({"cust": [1, 1, 2], "name": ["A", "A-dup", "B"]})

merged = orders.merge(customers, on="cust")

print("orders rows   :", len(orders))
print("customers rows:", len(customers))
print("merged rows   :", len(merged), "  <- MORE than we started with")
print()
print(merged)

orders rows   : 3
customers rows: 3
merged rows   : 5   <- MORE than we started with

   cust  amt   name
0     1   10      A
1     1   10  A-dup
2     1   20      A
3     1   20  A-dup
4     2   30      B


In [7]:
print("true total of amounts :", orders["amt"].sum())
print("total after the merge :", merged["amt"].sum(), "  <- inflated")

# cust 1 appears twice on EACH side, so 2 x 2 = 4 rows come out of it.
# No error. No warning. Every total computed afterwards is wrong.

true total of amounts : 60
total after the merge : 90   <- inflated


### Two defences

In [8]:
# 1. validate - state your assumption and let pandas check it
try:
    orders.merge(customers, on="cust", validate="many_to_one")
except Exception as e:
    print(type(e).__name__, ":", str(e)[:70])

MergeError : Merge keys are not unique in right dataset; not a many-to-one merge


In [9]:
# 2. assert on the row count - one line, catches everything
def safe_merge(left, right, **kwargs):
    """Merge, and refuse to silently change the row count."""
    out = left.merge(right, **kwargs)
    assert len(out) == len(left), f"row count changed: {len(left)} -> {len(out)}"
    return out


try:
    safe_merge(orders, customers, on="cust")
except AssertionError as e:
    print("AssertionError:", e)

# Check len(df) before and after EVERY merge. It costs one line.

AssertionError: row count changed: 3 -> 5


## 4.2 `concat` — stacking, not matching

In [10]:
jan = pd.DataFrame({"name": ["Ravi"], "mark": [88]})
feb = pd.DataFrame({"name": ["Sara"], "mark": [91]})

stacked = pd.concat([jan, feb])
print(stacked)
print("index:", stacked.index.tolist(), "  <- duplicated!")
print()
print("with ignore_index=True:")
print(pd.concat([jan, feb], ignore_index=True))

   name  mark
0  Ravi    88
0  Sara    91
index: [0, 0]   <- duplicated!

with ignore_index=True:
   name  mark
0  Ravi    88
1  Sara    91


In [11]:
# A missing column becomes NaN, not an error
other = pd.DataFrame({"name": ["Amit"], "city": ["Delhi"]})
print(pd.concat([jan, other], ignore_index=True))

   name  mark   city
0  Ravi  88.0    NaN
1  Amit   NaN  Delhi


In [14]:
# axis=1 glues columns side by side, aligning on the INDEX
print(pd.concat([jan, feb.reset_index(drop=True)], axis=1))

# It matches on the index, NOT on any key. Two tables whose rows are in
# different orders will be silently mismatched.
#
# If the rows mean something - a customer, a date - use merge, not concat.

   name  mark  name  mark
0  Ravi    88  Sara    91


> ### The one-line way to decide
>
> Do the two tables describe the **same things**, with different information about each?
> → **merge**, on a key.
>
> Do they describe **different things**, with the same information about each?
> → **concat**, stacking rows.
>
> Twelve monthly sales files are a `concat`. Sales plus a customer lookup is a `merge`.

---
# 5. Putting it together — three tables into one report

In [ ]:
# Build three input files
pd.DataFrame({"id": [1, 2, 3, 4], "name": ["Ravi", "Sara", "Amit", "Neha"],
              "city": ["Pune", "Mumbai", "Delhi", "Goa"]}).to_csv("students.csv", index=False)
pd.DataFrame({"id": [1, 2, 3, 5], "python": [88, 91, 45, 70],
              "stats": [71, 84, 38, 66]}).to_csv("marks.csv", index=False)
pd.DataFrame({"city": ["Pune", "Mumbai", "Delhi"],
              "state": ["MH", "MH", "DL"]}).to_csv("cities.csv", index=False)

students = pd.read_csv("students.csv")
marks    = pd.read_csv("marks.csv")
cities   = pd.read_csv("cities.csv")

# ---- 0. look BEFORE you join
check = students.merge(marks, on="id", how="outer", indicator=True)
print(check["_merge"].value_counts())

In [ ]:
# ---- 1. keep every student, even those with no marks
before = len(students)
df = students.merge(marks, on="id", how="left", validate="one_to_one")
assert len(df) == before, "the merge changed the row count!"

# ---- 2. add the state from a lookup table
df = df.merge(cities, on="city", how="left", validate="many_to_one")
assert len(df) == before

# ---- 3. report what is missing, do not hide it
print("students with no marks:", df["python"].isna().sum())
print("cities with no state  :", df["state"].isna().sum())
print()
print(df)

In [ ]:
# ---- 4. now it is safe to summarise
print(df.groupby("state")["python"].mean().round(1))

# Neha is in Goa, which is not in the lookup table, so she has no state
# and is excluded from this grouping - exactly the Day 17 behaviour.
# We know that BECAUSE we printed the missing count first.

Two `assert`s and one `indicator`. That is the difference between a report you can defend
and one you merely hope is right.

Note the two different `validate` values: `one_to_one` because each student appears once in
each table, and `many_to_one` because many students share a city but each city appears once
in the lookup.

---
# 6. Recap — the twelve things to remember

1. `merge` combines two tables on a shared key.
2. The real question is what happens to rows that **don't** match.
3. `inner` keeps matches only — and it is the **default**.
4. `left` keeps every left row, filling the rest with `NaN`.
5. `outer` keeps everything; `right` is rare — swap the tables.
6. An unmatched row turns an int column into float.
7. `indicator=True` adds `both` / `left_only` / `right_only`.
8. Run outer + indicator **first**, then pick your real join.
9. ⚠️ Duplicate keys multiply rows. 3 and 3 can give 5.
10. `validate="many_to_one"` states your assumption.
11. `assert len(after) == len(before)` after every merge.
12. `concat` stacks; use `ignore_index=True`. `axis=1` aligns on index.

---

### 📝 Now open **`Day18_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Take two CSVs and run all four joins, recording the row count of each.
- Deliberately duplicate a key and watch the row count explode.
- Rebuild Day 9's city lookup as a merge instead of a dictionary.

### Next class — Topic 1.18: Data cleaning — missing values & duplicates
Finding them, deciding what to do, and the cost of each choice.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*